Session spark


In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

spark = (
    SparkSession.builder
    .appName("ex4-weather-transform")
    .master("spark://spark-master:7077")
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

config + schéma

In [2]:
BOOTSTRAP = "kafka:9092"
SOURCE_TOPIC = "weather_stream"
TARGET_TOPIC = "weather_transformed"
CHECKPOINT = "/home/jovyan/work/checkpoints/ex4_weather_transformed"

WEATHER_SCHEMA = StructType([
    StructField("city", StringType()),
    StructField("country", StringType()),
    StructField("current", StructType([
        StructField("time", StringType()),
        StructField("temperature_2m", DoubleType()),
        StructField("wind_speed_10m", DoubleType()),
        StructField("weather_code", IntegerType()),
    ])),
])

la transformation

In [3]:
def transform(kafka_df):
    """colonnes kafka brutes -> une ligne météo propre avec ses niveaux d'alerte"""
    parsed = kafka_df.select(
        F.col("timestamp").alias("event_time"),
        F.from_json(F.col("value").cast("string"), WEATHER_SCHEMA).alias("data"),
    )

    weather = parsed.select(
        "event_time",
        F.col("data.current.temperature_2m").alias("temperature"),
        F.col("data.current.wind_speed_10m").alias("windspeed"),
        F.col("data.city").alias("city"),
        F.col("data.country").alias("country"),
        F.col("data.current.weather_code").alias("weather_code"),
    ).where(
        F.col("city").isNotNull()
        & F.col("temperature").isNotNull()
        & F.col("windspeed").isNotNull()
    )

    return (
        weather
        .withColumn(
            "wind_alert_level",
            F.when(F.col("windspeed") < 10, "level_0")
             .when(F.col("windspeed") <= 20, "level_1")
             .otherwise("level_2"),
        )
        .withColumn(
            "heat_alert_level",
            F.when(F.col("temperature") < 25, "level_0")
             .when(F.col("temperature") <= 35, "level_1")
             .otherwise("level_2"),
        )
    )


def to_kafka(df):
    """kafka attend une colonne 'value' (et optionnellement 'key')"""
    return df.select(
        F.col("city").alias("key"),
        F.to_json(F.struct(*df.columns)).alias("value"),
    )

In [4]:
batch = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", SOURCE_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)
batch.printSchema()
transform(batch).orderBy(F.desc("event_time")).show(10, truncate=False)

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

+-----------------------+-----------+---------+----------+--------------------+------------+----------------+----------------+
|event_time             |temperature|windspeed|city      |country             |weather_code|wind_alert_level|heat_alert_level|
+-----------------------+-----------+---------+----------+--------------------+------------+----------------+----------------+
|2026-11-10 15:51:55.069|15.5       |8.26     |Wellington|New Zealand         |3           |level_0         |level_0         |
|2026-11-10 15:51:55.042|6.6        |6.7      |Reykjavik |Iceland             |2           |level_0         |level_0         |
|2026-11-10 15:51:55.013|30.0       |2.82     |Dubai     |United Arab Emirates|0  

Lancer stream vers  Kafka

In [5]:
stream = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", SOURCE_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

query = (
    to_kafka(transform(stream))
    .writeStream.format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("topic", TARGET_TOPIC)
    .option("checkpointLocation", CHECKPOINT)
    .outputMode("append")
    .queryName("ex4_weather_transformed")
    .start()
)

Surveillance

In [7]:
print(query.status)
p = query.lastProgress
if p:
    print(f"batch {p['batchId']} : {p['numInputRows']} lignes lues à {p['timestamp']}")
    

{'message': 'Getting offsets from KafkaV2[Subscribe[weather_stream]]', 'isDataAvailable': False, 'isTriggerActive': True}
batch 12 : 4 lignes lues à 2026-11-10T15:53:51.139Z
